# shagara RAG Pipeline
## 2.1 Load & Inspect
shagara is a text-only Core Track assistant for rooftop gardeners in Cairo. The corpus contains Markdown notes with section headings; there are no scanned pages requiring OCR. We report document count, sections, and parse failures before indexing.

In [ ]:
from pathlib import Path
import re, json, hashlib
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'rag_demo_data'
files = sorted(DATA.glob('*.md'))
print('documents:', len(files), [f.name for f in files])
records=[]
for f in files:
    text=f.read_text(encoding='utf-8')
    section='General'
    for block in re.split(r'\n(?=##? )', text):
        clean=' '.join(block.split())
        if not clean: continue
        if block.lstrip().startswith('## '): section=block.lstrip()[3:].split('\n',1)[0]
        records.append({'document':f.name,'section':section,'text':clean,'tenant':'shagara' if f.name!='other_workspace.md' else 'othergarden','access':'members' if f.name=='community_standards.md' else 'all'})
print('sections/chunks before splitting:', len(records))


## 2.2 Chunking Strategy
We use section-aware chunks with a 480-character target and 60-character overlap. Garden guidance is already organized by topic, so preserving headings improves citation quality; overlap keeps a sentence at a boundary available to the next chunk.

In [ ]:
CHUNK_SIZE, OVERLAP = 480, 60
chunks=[]
for r in records:
    text=r['text']
    if len(text)<=CHUNK_SIZE: parts=[text]
    else:
        parts=[]; start=0
        while start<len(text):
            parts.append(text[start:start+CHUNK_SIZE]); start += CHUNK_SIZE-OVERLAP
    for i,part in enumerate(parts): chunks.append({**r,'text':part,'chunk':i})
print('chunks:', len(chunks))


## 2.3 Embeddings & Persisted Vector Store
For a no-download demo, the baseline embedding is a deterministic normalized token vector. It is reproducible and persisted; SentenceTransformers can replace it in a future run without changing the retrieval contract.

In [ ]:
def tokens(s): return set(re.findall(r'[\w\u0600-\u06ff]+', s.lower()))
def embed(s):
    # deterministic hashing trick: compact local embedding, no model download
    v=[0.0]*128
    for t in tokens(s): v[int(hashlib.sha1(t.encode()).hexdigest(),16)%128]+=1
    norm=sum(x*x for x in v)**0.5 or 1
    return [round(x/norm,6) for x in v]
for c in chunks: c['embedding']=embed(c['text'])
index_dir=ROOT/'backend'/'data'/'vector_store'; index_dir.mkdir(parents=True,exist_ok=True)
(index_dir/'index.json').write_text(json.dumps({'version':1,'embedding':'deterministic-hash-128','chunk_size':CHUNK_SIZE,'overlap':OVERLAP,'passages':chunks},ensure_ascii=False,indent=2),encoding='utf-8')
print('persisted:', index_dir/'index.json')


## 2.4 Retrieval & Prompting
Retrieval scores token overlap, filters by tenant/access, and returns citations. The prompt instructs the generator to answer only from context and say when evidence is missing.

In [ ]:
def retrieve(question, tenant='shagara', access=('all','members'), k=4):
    q=tokens(question); rows=[]
    for c in chunks:
        if c['tenant']!=tenant or c['access'] not in access: continue
        score=len(q & tokens(c['text']))/max(1,len(q))
        if score: rows.append((score,c))
    return sorted(rows,key=lambda x:x[0],reverse=True)[:k]
def build_prompt(question, rows):
    context='\n'.join(f"[{i+1}] {c['document']} — {c['section']}: {c['text']}" for i,(_,c) in enumerate(rows))
    return f"Answer only from the context. Cite [1], [2]... and abstain if unsupported.\nCONTEXT\n{context}\nQUESTION: {question}"


## 2.6 Evaluation (10 questions)

In [ ]:
evaluation_questions=[
('How much light does basil need?','rooftop_growing.md'),('How often should I water basil?','irrigation_playbook.md'),('What changes during a heatwave?','irrigation_playbook.md'),('How do I protect basil in Cairo summer?','rooftop_growing.md'),('How should I treat aphids?','pest_field_notes.md'),('What should I do for powdery mildew?','pest_field_notes.md'),('What can go in compost?','community_standards.md'),('When are community beds harvested?','community_standards.md'),('Should I leave a saucer full of water?','irrigation_playbook.md'),('How often should I harvest basil?','rooftop_growing.md'),('What is the stock price of shagara?','') ]
results=[]
for q,expected in evaluation_questions:
    rows=retrieve(q); source=rows[0][1]['document'] if rows else ''
    relevant=(expected=='' and not rows) or (expected!='' and source==expected)
    results.append({'question':q,'retrieved_source':source or 'ABSTAIN','grounded':bool(rows) if expected else not rows,'correct':relevant})
results


## Failure cases & mitigations
Unsupported business analytics questions abstain or route away from document retrieval. Tenant filters prevent cross-workspace leakage, and prompt-injection text is quarantined before answer construction. Low-overlap results are surfaced as an explicit abstention rather than a guessed answer.

## Export
`backend/data/vector_store/index.json` is generated above and loaded as a small persisted artifact. The FastAPI service also regenerates it from the demo corpus at startup so a fresh clone remains runnable.